# Modeling

In this section, we extend our preprocessing pipeline to train and evaluate classification algorithms. Our goal is to identify the best-performing model for predicting customer churn.

## Imports and splitting data

Firstly, we will use our data preprocessing function and split our data for training and test part. The important thing in splitting will be `stratify=y`. This parameter will ensure that in training and test set there will be the same propotion of each class.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from utils import split_columns, data_preprocessing

X, y, pre_pipeline = data_preprocessing()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Model configuration

We will create list of dictionaries containing models and parameters grid, which will be used for hyperparameter tuning 

In [ ]:
models_config = [
    {
        'name': 'Support Vector Machine (SVM)',
        'model': SVC(probability=True, random_state=42),
        'params': {
            'classifier__C': [1],
            'classifier__kernel': ['linear']
        }
    }
]

results_data = []

print("\nRozpoczynam Grid Search z 5-krotną walidacją krzyżową...\n")

for config in models_config:
    print(f"--- Trenowanie: {config['name']} ---")

    full_pipeline = Pipeline(steps=pre_pipeline.steps + [('classifier', config['model'])])
    
    grid_search = GridSearchCV(
        estimator=full_pipeline,
        param_grid=config['params'],
        cv=5,              
        scoring='accuracy', 
        n_jobs=-1,          
        verbose=1
    )
    

    grid_search.fit(X_train, y_train)
    

    best_model = grid_search.best_estimator_
    

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, "predict_proba") else [0]*len(y_test)
    

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    

    results_data.append({
        'Model': config['name'],
        'Best Params': grid_search.best_params_,
        'Accuracy': acc,
        'F1-Score': f1,
        'ROC-AUC': auc
    })
    
    print(f"Najlepsze parametry: {grid_search.best_params_}")
    print(f"Accuracy na teście: {acc:.4f}\n")


results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values(by='Accuracy', ascending=False)

print("="*60)
print("PODSUMOWANIE WYNIKÓW (Zbiór Testowy)")
print("="*60)

print(results_df[['Model', 'Accuracy', 'F1-Score', 'ROC-AUC']].to_markdown(index=False, floatfmt=".4f"))




Rozpoczynam Grid Search z 5-krotną walidacją krzyżową...

--- Trenowanie: Support Vector Machine (SVM) ---
Fitting 5 folds for each of 6 candidates, totalling 30 fits


KeyboardInterrupt: 